In [17]:
import pandas as pd
import geopandas as gpd
from geopy.distance import geodesic
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 로드
store_df = pd.read_csv("data/소상공인시장진흥공단_상가(상권)정보_서울_202412.csv", encoding="utf-8")
facility_df = pd.read_csv("data/서울시 상권분석서비스(집객시설-상권배후지).csv", encoding="cp949")
market_area_gdf = gpd.read_file("data/서울시 상권분석서비스(영역-상권).shp")
mapping_df = pd.read_csv("data/카페_상권_매핑_데이터.csv", encoding="cp949")

# 2. 카페 필터링
cafe_df = store_df[store_df['상권업종소분류명'].str.contains('카페', na=False)].copy()
cafe_df = cafe_df.dropna(subset=['위도', '경도'])
cafe_df = cafe_df.merge(mapping_df[['상가업소번호', 'TRDAR_CD']], on='상가업소번호', how='left')

# 3. 교통시설 수 계산
facility_df['교통시설_수'] = (
    facility_df['공항_수'].fillna(0) +
    facility_df['철도_역_수'].fillna(0) +
    facility_df['버스_터미널_수'].fillna(0) +
    facility_df['지하철_역_수'].fillna(0) +
    facility_df['버스_정거장_수'].fillna(0)
)
traffic_count = facility_df.groupby('상권배후지_코드')[['교통시설_수']].sum().reset_index()

# 4. shapefile 중심점 계산 (좌표계 보정)
market_area_gdf = market_area_gdf.to_crs(epsg=4326)

projected = market_area_gdf.to_crs(epsg=5179)
projected['centroid'] = projected.geometry.centroid
centroid_gdf = gpd.GeoDataFrame(geometry=projected['centroid'], crs=5179).to_crs(epsg=4326)
projected['center_lat'] = centroid_gdf.geometry.y
projected['center_lon'] = centroid_gdf.geometry.x
market_centroids = projected[['TRDAR_CD', 'center_lat', 'center_lon']]

# 5. 카페 좌표 → GeoDataFrame
cafe_gdf = gpd.GeoDataFrame(
    cafe_df.merge(mapping_df[['상가업소번호', 'TRDAR_CD']], on='상가업소번호', how='left'),
    geometry=gpd.points_from_xy(cafe_df['경도'], cafe_df['위도']),
    crs='EPSG:4326'
)

# 6. 카페 ↔ 상권 공간 조인
cafe_with_area = gpd.sjoin(cafe_gdf, market_area_gdf[['TRDAR_CD', 'geometry']], how='left', predicate='within')

# 7. 거리 계산
cafe_with_area = cafe_with_area.merge(market_centroids, on='TRDAR_CD', how='left')
def compute_distance(row):
    if pd.isna(row['center_lat']) or pd.isna(row['center_lon']):
        return None
    return geodesic((row['위도'], row['경도']), (row['center_lat'], row['center_lon'])).meters
cafe_with_area['상권중심_거리'] = cafe_with_area.apply(compute_distance, axis=1)

# 8. 층수 추출
def extract_floor_from_column(floor_str):
    try:
        if isinstance(floor_str, str):
            if '지하' in floor_str:
                return -1
            digits = ''.join(filter(str.isdigit, floor_str))
            return int(digits) if digits else 1
        elif pd.isna(floor_str):
            return 1  # 결측치는 기본 1층 처리
        else:
            return 1
    except:
        return 1

# 층정보 컬럼에서 직접 층수 추출
cafe_with_area['층수'] = cafe_with_area['층정보'].apply(extract_floor_from_column)

# 9. 교통시설 수 병합 (상권 코드 매핑)
# TRDAR_CD와 상권배후지_코드가 같다고 가정
traffic_count = traffic_count.rename(columns={'상권배후지_코드': 'TRDAR_CD'})
cafe_with_area['TRDAR_CD'] = cafe_with_area['TRDAR_CD'].astype(str)
traffic_count['TRDAR_CD'] = traffic_count['TRDAR_CD'].astype(str)
cafe_final = cafe_with_area.merge(traffic_count, on='TRDAR_CD', how='left')
cafe_final['교통시설_수'] = cafe_final['교통시설_수'].fillna(0)

# 10. 정규화 및 점수 계산
scaler = MinMaxScaler()
cafe_final[['상권중심_거리_norm']] = scaler.fit_transform(cafe_final[['상권중심_거리']])
cafe_final['층수_norm'] = scaler.fit_transform(cafe_final[['층수']])
cafe_final['교통시설_수_norm'] = scaler.fit_transform(cafe_final[['교통시설_수']])

# 11. 최종 접근성 점수 계산
cafe_final['접근성점수'] = (
    0.4 * (1 - cafe_final['상권중심_거리_norm']) +  # 중심과 가까울수록 좋음
    0.3 * cafe_final['교통시설_수_norm'] +         # 교통시설이 많을수록 좋음
    0.3 * (1 - cafe_final['층수_norm'])             # 1층일수록 좋음
)

# 결과 저장 or 출력
cafe_result = cafe_final[['상호명', '도로명주소', 'TRDAR_CD', '교통시설_수', '상권중심_거리', '층수', '접근성점수']]
cafe_result.to_csv("카페_교통접근성_분석결과.csv", index=False, encoding='utf-8-sig')
print("접근성 분석 완료. 결과 저장됨: 카페_교통접근성_분석결과.csv")

접근성 분석 완료. 결과 저장됨: 카페_교통접근성_분석결과.csv


In [18]:
print("카페 좌표계:", cafe_gdf.crs)
print("상권 좌표계:", projected[['TRDAR_CD', 'geometry']].crs)

print("카페 수:", len(cafe_gdf))
print("공간조인 후 상권 매핑된 카페 수:", cafe_with_area['TRDAR_CD'].notna().sum())

카페 좌표계: EPSG:4326
상권 좌표계: EPSG:5179
카페 수: 25916
공간조인 후 상권 매핑된 카페 수: 26618


In [19]:
nan_rows = cafe_final[cafe_final['접근성점수'].isna()]
print("NaN인 샘플 수:", len(nan_rows))

# 어떤 항목이 NaN인지 확인
print(nan_rows[['상호명', '상권중심_거리', '상권중심_거리_norm', '교통시설_수', '교통시설_수_norm', '층수', '층수_norm']].head(10))

NaN인 샘플 수: 5800
                           상호명  상권중심_거리  상권중심_거리_norm  교통시설_수  교통시설_수_norm  \
11                성균관대학구내학생휴게실      NaN           NaN     0.0          0.0   
45                       샌드프레소      NaN           NaN     0.0          0.0   
46                        산모퉁이      NaN           NaN     0.0          0.0   
51                   컴포즈커피동묘롯데      NaN           NaN     0.0          0.0   
77   ThePianowasdrinking，notme      NaN           NaN     0.0          0.0   
79                       이디야커피      NaN           NaN     0.0          0.0   
81                        지유명차      NaN           NaN     0.0          0.0   
82                        커피홀릭      NaN           NaN     0.0          0.0   
92                        베스트빈      NaN           NaN     0.0          0.0   
136                      이디야커피      NaN           NaN     0.0          0.0   

     층수  층수_norm  
11    1      0.0  
45    1      0.0  
46    1      0.0  
51    1      0.0  
77    1      0.0  
79    1    